# GPT-2 파인튜닝

| 항목                     | `distilbert-base-uncased` | `gpt2`                    |
| ---------------------- | ------------------------- | ------------------------- |
| 🔢 모델 타입               | Encoder (BERT 계열)         | Decoder (GPT 계열)          |
| 📦 사전학습 목적             | 마스킹된 언어 모델(Masked LM)     | 오토리그레시브 언어 생성(Next token) |
| 🧠 파라미터 수              | 약 66M                     | 약 124M                    |
| 🏗️ 레이어 수              | 6                         | 12                        |
| 🧩 히든 크기 (Hidden Size) | 768                       | 768                       |
| 🔁 어텐션 헤드 수            | 12                        | 12                        |
| 📏 최대 시퀀스 길이           | 512 tokens                | 1024 tokens               |
| 📄 토크나이저 타입            | WordPiece (BERT형)         | Byte-Pair Encoding (BPE)  |
| 💬 생성 능력               | ❌ (비생성형, 분류에 적합)          | ✅ (텍스트 생성에 최적)            |
| ⚙️ 사용 예시               | 텍스트 분류, 감정 분석 등           | 텍스트 생성, 요약, 질문응답 등        |
| ⚡ 연산 효율성               | 빠름 (BERT의 경량화 버전)         | 느림 (텍스트 생성 연속 수행 필요)      |



# GPT-2 파인튜닝, 전체 흐름 한눈에 보기

## 🎯 목표
GPT-2에게 **우리만의 질문-답변 방식**을 가르쳐서, 원하는 스타일로 대답하게 만드는 것!

## 🛠️ 사용하는 기술
- **GPT-2**: 텍스트 생성에 특화된 언어 모델
- **LoRA**: 모델 전체를 바꾸지 않고 일부만 효율적으로 조정하는 기술
- **Trainer**: 학습을 쉽게 해주는 자동화 도구

## 📋 전체 단계 요약

1. **도구 설치**: 필요한 라이브러리 불러오기
2. **모델 로드**: GPT-2와 토크나이저 불러오기
3. **LoRA 설정**: 빠르고 가볍게 학습할 수 있도록 구성
4. **데이터 준비**: 질문-답변 예시 입력
5. **토큰화**: 텍스트 → 숫자 변환
6. **학습 설정**: 반복 횟수, 배치 크기 등 지정
7. **학습 실행**: 모델에게 예시 보여주며 훈련
8. **결과 확인**: 새 질문 넣어보기

## 💡 핵심 요약
- GPT-2는 원래 똑똑함 → 우리는 "내 스타일(내데이터)"로 조금만 바꿈
- LoRA 덕분에 가볍고 빠르게 가능



In [1]:
# STEP 1: 환경 설정
# uv add transformers datasets peft accelerate bitsandbytes

In [2]:
# ✅ STEP 2: 기본 라이브러리 임포트
# 모델 학습에 필요한 필수 라이브러리들을 불러옵니다.
import torch  # PyTorch: 딥러닝 프레임워크
from datasets import Dataset  # 텍스트 데이터를 Dataset 객체로 변환하는 데 사용
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import get_peft_model, LoraConfig, TaskType  # LoRA 기반 파인튜닝 지원

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [3]:
from huggingface_hub import login
from dotenv import load_dotenv
import os

# .env 파일 로드
load_dotenv(override=True)

HF_TOKEN = os.getenv("HF_TOKEN")
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
# ✅ STEP 3: 모델 및 토크나이저 로드
# Hugging Face에서 제공하는 GPT-2 모델과 토크나이저를 불러옵니다.

model_id = "gpt2"  # 사용할 모델 지정 (GPT-2)

# 토크나이저는 텍스트를 숫자 시퀀스로 바꿔주는 도구입니다.
tokenizer = AutoTokenizer.from_pretrained(model_id)

# ⚠️ 수정: GPT-2는 pad_token이 없어서 보통 eos_token을 그대로 pad_token으로 재사용하는데,
# 이렇게 하면 DataCollatorForLanguageModeling이 "pad 토큰 위치는 loss 계산에서 제외"할 때
# 진짜 EOS(문장 종료) 토큰까지 같이 제외돼서, 모델이 "답변 후 멈춰야 한다"는 것을 전혀 배우지 못합니다.
# 그래서 EOS와 id가 겹치지 않는 별도의 PAD 토큰을 새로 추가합니다.
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})

# 사전학습된 GPT-2 모델을 불러옵니다. 텍스트 생성 작업에 최적화된 모델입니다.
model = AutoModelForCausalLM.from_pretrained(model_id)

# 새로 추가한 [PAD] 토큰만큼 임베딩 테이블 크기를 늘려줍니다.
# (반드시 LoRA를 적용하기 "전"에 해야 합니다.)
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [5]:
# ✅ STEP 4: LoRA 설정
# [포인트] LoRA(Low-Rank Adaptation)는 대규모 모델을 가볍게 미세조정할 수 있게 해주는 기술입니다.
# 전체 모델을 학습시키지 않고, 일부 작은 파라미터만 추가로 학습하기 때문에 훨씬 효율적입니다.

lora_config = LoraConfig(
    r=16,  # 랭크: LoRA 내부 차원. 작을수록 가볍고 빠름
    lora_alpha=32,  # ⚠️ 수정: alpha < r 이면 LoRA 업데이트가 (alpha/r)배로 축소되어 거의 학습이 안 됩니다. 보통 alpha >= r로 설정합니다.
    target_modules=["c_attn", "c_proj", "q_attn", "mlp.c_fc", "mlp.c_proj"],  # ⚠️ 수정: MLP 레이어도 포함해서 학습 용량을 늘립니다.
    # 🔍 GPT-2의 레이어들
    # - "c_attn": 쿼리, 키, 밸류를 생성하는 핵심 어텐션 입력 레이어
    # - "c_proj": 어텐션 출력 벡터를 변환하는 투영 레이어
    # - "q_attn": self-attention 중 쿼리 연산에 관여하는 부분 (GPT-J 등 일부 구조에 존재)
    # - "mlp.c_fc": 피드포워드 네트워크의 첫 번째 선형 계층 (MLP의 입력 부분)
    # - "mlp.c_proj": MLP의 출력 부분
    lora_dropout=0.05,  # 드롭아웃 적용 (과적합 방지)
    bias="none",  # bias 파라미터는 학습하지 않음
    task_type=TaskType.CAUSAL_LM,  # 작업 유형: 언어 생성 (Causal Language Modeling)
)

# ✅ 모델에 LoRA 설정을 적용합니다.
# 기존 GPT-2 위에 LoRA 구조를 얹어, 일부 파라미터만 학습 가능하게 만듭니다.
# 추론 시에는 원래 모델과 LoRA가 함께 사용되므로 merge 없이도 작동합니다.
model = get_peft_model(model, lora_config)

# 실제로 학습 가능한 파라미터 수를 출력해봅니다 (LoRA 파라미터만 학습되므로 매우 적습니다).
model.print_trainable_parameters()

trainable params: 2,359,296 || all params: 126,799,872 || trainable%: 1.8606


/mnt/e/gg_ai_merbership_1th/qlora_ft_ex/.venv/lib/python3.12/site-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [6]:
# ✅ STEP 5: 간단한 학습 데이터셋 정의
# 질문-답변 형태의 짧은 데이터셋을 정의합니다.
# ⚠️ 수정: 2개 문장만으로는 "질문-답변 형식" 자체를 일반화해서 배우기 어렵고, 그냥 암기 수준에 그칩니다.
# 실습 목적이라도 형식을 배울 수 있도록 예시를 몇 개 더 추가합니다.
data = {
    "text": [
        "### 질문: joy강사의 별명은?\n### 답변: 스마일",
        "### 질문: 바다는 왜 파란가요?\n### 답변: 햇빛의 산란",
        "### 질문: 오늘의 날씨는?\n### 답변: 맑음",
        "### 질문: GPT-2는 어떤 모델인가요?\n### 답변: 디코더 기반의 텍스트 생성 모델입니다",
        "### 질문: LoRA란 무엇인가요?\n### 답변: 모델 일부만 효율적으로 학습하는 기법입니다",
        "### 질문: 파이썬은 무엇에 쓰이나요?\n### 답변: 다양한 분야에 쓰이는 프로그래밍 언어입니다",
    ]
}
# Hugging Face Dataset 객체로 변환
dataset = Dataset.from_dict(data)


In [7]:
# ✅ STEP 6: 토크나이즈 함수
# [포인트] 텍스트 데이터를 모델에 넣기 위해 숫자 형태(토큰 ID)로 변환합니다.
# ⚠️ 수정 1: 문장 끝에 EOS 토큰을 명시적으로 붙여서, 모델이 "답변 후 멈춰야 한다"는 것을 학습하게 합니다.
# ⚠️ 수정 2: padding="max_length"로 미리 128까지 채우지 않고, 배치를 묶을 때
#           데이터 콜레이터가 배치별로 동적 패딩하도록 둡니다. (짧은 문장을 억지로 채워봐야 낭비입니다.)

def tokenize_function(example):
    text = example["text"] + tokenizer.eos_token
    return tokenizer(text, truncation=True, max_length=64)

# 데이터셋에 토크나이즈 함수를 적용하여 숫자형 시퀀스로 변환
tokenized_dataset = dataset.map(tokenize_function)


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

In [8]:
# ✅ STEP 7: 데이터 콜레이터
# [개념] 데이터 콜레이터는 배치로 묶을 때 패딩, 마스킹 등을 자동으로 처리해주는 도구입니다.
# GPT처럼 다음 단어를 예측하는 방식에서는 MLM(False)로 설정합니다.

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


데이터 콜레이터란?
DataCollatorForLanguageModeling은 모델에 데이터를 넣기 직전에 문장들을 자동으로 정리해주는 도구입니다.
모델이 문장을 한 번에 여러 개 처리하려면 길이를 맞춰야 하는데, 그 과정을 **패딩(padding)**이라고 해요.
또, 어떤 위치를 학습해야 할지 표시해주는 **마스킹(masking)**도 필요하죠.

이 과정을 직접 처리하지 않아도 되게끔, 데이터 콜레이터가 알아서 처리해 줍니다.

In [9]:
# ✅ STEP 8: 트레이닝 설정

# [포인트] 모델 학습에 필요한 하이퍼파라미터들을 설정합니다.
# ⚠️ 수정: LoRA는 전체 파라미터의 일부(1~2%)만 학습하기 때문에 기본 학습률(5e-5)로는
# 데이터를 다 도는 step 수가 적을 때 거의 학습이 안 됩니다. LoRA는 보통 Full FT보다
# 높은 학습률(1e-4 ~ 3e-4)과 더 많은 epoch이 필요합니다.
training_args = TrainingArguments(
    output_dir="./results",  # 결과 디렉토리
    per_device_train_batch_size=10,  # GPU나 CPU 1개당 배치 사이즈
    num_train_epochs=300,  # ⚠️ 수정: 데이터가 늘고 LoRA는 수렴이 느려서 epoch을 늘립니다
    learning_rate=2e-4,  # ⚠️ 수정: LoRA 파라미터 학습을 위해 학습률을 높입니다
    logging_steps=30,  # 30스텝마다 로그 출력 (학습 진행 상황 확인)
    save_strategy="no",  # 중간 저장 생략 (데모 목적이므로)
    fp16=False,  # GPU 없이 CPU로 학습 시 False 설정,
         report_to = "none",  # wandb, tensorboard 등 외부 로깅 툴 비활성화

)


✅ step과 epoch의 차이 정리

- **step은 iteration과 동일한 의미**입니다. 즉, 한 번의 파라미터 업데이트를 뜻합니다.
- 모델이 **한 배치(batch)**를 보고 학습하는 것이 1 step (1 iteration)입니다.
- 반면 **epoch**은 전체 데이터를 한 바퀴 학습한 것을 의미합니다.
- 예: 데이터 100개, 배치 크기 10이면 → 10 step = 1 epoch입니다.
- 정리하면, **여러 step이 모여 하나의 epoch을 구성**합니다.


In [10]:
# ✅ STEP 9: 트레이너 설정 및 학습 시작
# Hugging Face의 Trainer 클래스를 사용해 학습을 진행합니다.
# 위에서 정의한 모델, 데이터, 설정값을 모두 전달하여 학습을 시작합니다.\
import numpy as np
# 정확도를 계산하는 함수 정의
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # padding 부분은 무시
    mask = labels != -100
    correct = (predictions == labels) & mask
    accuracy = correct.sum() / mask.sum()

    return {"accuracy": accuracy}
trainer = Trainer(
    model=model,  # 학습할 모델
    args=training_args,  # 학습 설정
    train_dataset=tokenized_dataset,  # 학습 데이터
    # tokenizer=tokenizer,  # 텍스트 디코딩용   # Transformers 4.x 문법
    processing_class=tokenizer,  # 텍스트 디코딩용, Transformers 5.x 문법
    data_collator=data_collator,  # 배치 구성 도우미
    compute_metrics=compute_metrics  # <-- 여기 추가

)

# 학습 시작!
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50257}.


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
30,2.631150
60,1.209778
90,0.564097
120,0.301339
150,0.197324
180,0.151454
210,0.119961
240,0.110812
270,0.106577
300,0.102481


TrainOutput(global_step=300, training_loss=0.5494973540306092, metrics={'train_runtime': 20.3145, 'train_samples_per_second': 88.607, 'train_steps_per_second': 14.768, 'total_flos': 60421452595200.0, 'train_loss': 0.5494973540306092, 'epoch': 300.0})

In [11]:
# joy강사의 별명을 묻는 질문에 답을 하도록 구성합니다.
# ⚠️ 수정: 학습 데이터와 동일한 포맷(콜론 뒤 공백 위치 등)으로 맞춰줍니다.
#         포맷이 학습 때와 다르면 토큰화 결과가 달라져서 모델이 학습한 패턴과 잘 안 맞습니다.
input_text = "### 질문: joy강사의 별명은?\n### 답변:"

# ⚠️ 수정: trainer.train() 이후 모델이 학습(train) 모드로 남아 있으면 LoRA의 dropout(0.05)이
# 추론 때도 계속 적용되어 결과가 흔들립니다. 반드시 eval() 모드로 바꿔줍니다.
model.eval()

# 입력 문장을 숫자로 바꿔주는 tokenizer를 사용해 모델이 이해할 수 있는 형식으로 변환합니다.
# return_tensors="pt"는 PyTorch 텐서 형태로 반환하겠다는 뜻입니다.
inputs = tokenizer(input_text, return_tensors="pt")

# 모델이 올라가 있는 디바이스(GPU 또는 CPU)를 가져옵니다.
device = model.device

# 입력 데이터도 모델이 있는 디바이스로 옮겨줍니다.
# 그래야 모델과 데이터가 같은 장치에 있어 연산이 가능합니다.
inputs = {k: v.to(device) for k, v in inputs.items()}

# 모델에게 답변을 생성하도록 지시합니다.
# max_new_tokens=50은 최대 50개의 새로운 단어(토큰)를 생성하겠다는 의미입니다.
# ⚠️ 수정: do_sample=False로 그리디 디코딩을 명시하고, eos_token_id/pad_token_id를 지정하고,
#         repetition_penalty와 no_repeat_ngram_size로 문구 반복을 막아줍니다.
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
    )

# 생성된 답변을 사람이 읽을 수 있는 문자열로 바꿔서 출력합니다.
# skip_special_tokens=True는 시작/종료 같은 특수 기호는 출력하지 않겠다는 의미입니다.
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


### 질문: joy강사의 별명은?
### 답변: 햇빛의 산란


# GPT2파인튜닝 Full FineTuning(lora 제거)

In [12]:
# STEP 1: 환경 설정
# uv add transformers datasets


In [13]:
# STEP 2: 기본 라이브러리 임포트
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

# STEP 3: 모델 및 토크나이저 로드
model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# ⚠️ 수정: eos_token을 그대로 pad_token으로 쓰면, 데이터 콜레이터가 pad 위치를 -100으로
# 마스킹할 때 진짜 EOS까지 같이 지워져서 "답변 후 멈추기"를 학습할 수 없습니다.
# id가 겹치지 않는 별도의 PAD 토큰을 추가합니다.
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})

model = AutoModelForCausalLM.from_pretrained(model_id)
model.resize_token_embeddings(len(tokenizer))  # 새로 추가한 [PAD] 토큰 반영
model.config.pad_token_id = tokenizer.pad_token_id

# STEP 4: 학습 데이터 정의
# ⚠️ 수정: 예시를 더 추가해서 "질문-답변 형식" 자체를 배울 수 있도록 합니다.
data = {
    "text": [
        "### 질문: joy강사의 별명은?\n### 답변: smile",
        "### 질문: 오늘의 날씨는?\n### 답변: 맑음",
        "### 질문: 바다는 왜 파란가요?\n### 답변: 햇빛의 산란",
        "### 질문: GPT-2는 어떤 모델인가요?\n### 답변: 디코더 기반의 텍스트 생성 모델입니다",
        "### 질문: LoRA란 무엇인가요?\n### 답변: 모델 일부만 효율적으로 학습하는 기법입니다",
        "### 질문: 파이썬은 무엇에 쓰이나요?\n### 답변: 다양한 분야에 쓰이는 프로그래밍 언어입니다",
    ]
}
dataset = Dataset.from_dict(data)

# STEP 5: 토큰화
# ⚠️ 수정: EOS를 텍스트 끝에 명시적으로 붙이고, max_length=1024로 억지로 채우지 않고
# (짧은 문장을 1024까지 채우는 건 심한 낭비입니다) 콜레이터가 배치별로 동적 패딩하게 둡니다.
def tokenize_function(example):
    text = example["text"] + tokenizer.eos_token
    return tokenizer(text, truncation=True, max_length=64)

tokenized_dataset = dataset.map(tokenize_function)

# STEP 6: 데이터 콜레이터
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# STEP 7: 트레이닝 설정
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=10,
    num_train_epochs=50,
    logging_steps=1,
    save_strategy="no",
    fp16=False,
    report_to="none",  # wandb, tensorboard 등 외부 로깅 툴 비활성화

)

import numpy as np

# STEP 8: 트레이너 설정 및 학습


# 정확도를 계산하는 함수 정의
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # padding 부분은 무시
    mask = labels != -100
    correct = (predictions == labels) & mask
    accuracy = correct.sum() / mask.sum()

    return {"accuracy": accuracy}
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics # <-- 여기 추가

)

trainer.train()


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50257}.


Step,Training Loss
1,3.581772
2,3.259999
3,3.119754
4,2.708695
5,2.524972
6,2.402883
7,2.254396
8,1.981980
9,1.760630
10,1.739356


TrainOutput(global_step=50, training_loss=1.1037544286251069, metrics={'train_runtime': 4.8309, 'train_samples_per_second': 62.1, 'train_steps_per_second': 10.35, 'total_flos': 9798451200000.0, 'train_loss': 1.1037544286251069, 'epoch': 50.0})

In [14]:

# STEP 9: 추론 예시
input_text = "### 질문: joy강사의 별명은?\n### 답변:"
inputs = tokenizer(input_text, return_tensors="pt")

# 모델과 같은 디바이스로 이동
device = model.device
inputs = {k: v.to(device) for k, v in inputs.items()}

inputs


{'input_ids': tensor([[21017, 23821,   100,   230,   167,   105,   116,    25,  8716,   166,
            108,   243,   168,  8955, 35975,   246, 31619,   111,   226,   167,
            103,   227, 35975,   222,    30,   198, 21017, 31619,   233,   113,
            167,   111,   222,    25]], device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [15]:
# ⚠️ 수정: 추론 전 eval() 모드로 전환 + do_sample=False(그리디) 명시
# + eos_token_id/pad_token_id 명시 + repetition_penalty, no_repeat_ngram_size로
# 답변이 끝나지 않고 같은 문구를 계속 반복하는 것을 막아줍니다.
model.eval()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
    )
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### 질문: joy강사의 별명은?
### 답변: smile
